# CineSense-AI: SQL Database & Analysis

## Objective
This notebook focuses on storing and analyzing the processed MovieLens data using SQL.

The goal is to apply SQL concepts such as database creation, JOINs, aggregations, CTEs, subqueries, and window functions to extract meaningful insights that can later support the CineSense-AI recommendation system.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving movies_final.csv to movies_final.csv
Saving ratings_clean.csv to ratings_clean.csv
Saving tags_clean.csv to tags_clean.csv


In [ ]:
import pandas as pd
import sqlite3

# Load processed datasets
movies_df = pd.read_csv("movies_final.csv")
ratings_df = pd.read_csv("ratings_clean.csv")
tags_df = pd.read_csv("tags_clean.csv")

# Create SQLite database connection
conn = sqlite3.connect("cinesense.db")

print("Database connection created successfully!")

Database connection created successfully!


In [ ]:
# Store DataFrames as SQL tables

movies_df.to_sql(
    "movies",
    conn,
    if_exists="replace",
    index=False
)

ratings_df.to_sql(
    "ratings",
    conn,
    if_exists="replace",
    index=False
)

tags_df.to_sql(
    "tags",
    conn,
    if_exists="replace",
    index=False
)

print("Tables created successfully!")

Tables created successfully!


In [ ]:
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table';
"""

tables = pd.read_sql_query(query, conn)

display(tables)

,name
0,movies
1,ratings
2,tags


In [ ]:
query = """
SELECT
    (SELECT COUNT(*) FROM movies) AS total_movies,
    (SELECT COUNT(*) FROM ratings) AS total_ratings,
    (SELECT COUNT(*) FROM tags) AS total_tags;
"""

result = pd.read_sql_query(query, conn)

display(result)

,total_movies,total_ratings,total_tags
0,9742,100836,3683


## 1. Most Popular Movies

This analysis identifies the most popular movies based on the total number of user ratings received.

In [ ]:
query = """
SELECT
    movieId,
    clean_title,
    year,
    rating_count,
    average_rating
FROM movies
ORDER BY rating_count DESC
LIMIT 10;
"""

popular_movies = pd.read_sql_query(query, conn)

display(popular_movies)

,movieId,clean_title,year,rating_count,average_rating
0,356,Forrest Gump,1994.0,329,4.16
1,318,"Shawshank Redemption, The",1994.0,317,4.43
2,296,Pulp Fiction,1994.0,307,4.20
3,593,"Silence of the Lambs, The",1991.0,279,4.16
4,2571,"Matrix, The",1999.0,278,4.19
5,260,Star Wars: Episode IV - A New Hope,1977.0,251,4.23
6,480,Jurassic Park,1993.0,238,3.75
7,110,Braveheart,1995.0,237,4.03
8,589,Terminator 2: Judgment Day,1991.0,224,3.97
9,527,Schindler's List,1993.0,220,4.22


## 2. Top-Rated Movies with Minimum Rating Threshold

To avoid movies with very few ratings dominating the rankings, only movies with at least 50 user ratings are considered.

In [ ]:
query = """
SELECT
    movieId,
    clean_title,
    year,
    rating_count,
    average_rating
FROM movies
WHERE rating_count >= 50
ORDER BY average_rating DESC, rating_count DESC
LIMIT 10;
"""

top_rated_movies = pd.read_sql_query(query, conn)

display(top_rated_movies)

,movieId,clean_title,year,rating_count,average_rating
0,318,"Shawshank Redemption, The",1994.0,317,4.43
1,858,"Godfather, The",1972.0,192,4.29
2,2959,Fight Club,1999.0,218,4.27
3,750,Dr. Strangelove or: How I Learned to Stop Worr...,1964.0,97,4.27
4,1276,Cool Hand Luke,1967.0,57,4.27
5,1221,"Godfather: Part II, The",1974.0,129,4.26
6,904,Rear Window,1954.0,84,4.26
7,1213,Goodfellas,1990.0,126,4.25
8,48516,"Departed, The",2006.0,107,4.25
9,50,"Usual Suspects, The",1995.0,204,4.24


## 3. Most Active Users

This analysis identifies users with the highest number of movie ratings, providing insights into user engagement.

In [ ]:
query = """
SELECT
    userId,
    COUNT(*) AS total_ratings,
    ROUND(AVG(rating), 2) AS average_rating_given
FROM ratings
GROUP BY userId
ORDER BY total_ratings DESC
LIMIT 10;
"""

active_users = pd.read_sql_query(query, conn)

display(active_users)

,userId,total_ratings,average_rating_given
0,414,2698,3.39
1,599,2478,2.64
2,474,2108,3.40
3,448,1864,2.85
4,274,1346,3.24
5,610,1302,3.69
6,68,1260,3.23
7,380,1218,3.67
8,606,1115,3.66
9,288,1055,3.15


## 4. Movie Rating Analysis Using SQL JOIN

The movies and ratings tables are joined using `movieId` to calculate rating statistics directly from individual user-rating records.

In [ ]:
query = """
SELECT
    m.movieId,
    m.clean_title,
    m.year,
    COUNT(r.rating) AS total_ratings,
    ROUND(AVG(r.rating), 2) AS avg_rating
FROM movies m
JOIN ratings r
    ON m.movieId = r.movieId
GROUP BY
    m.movieId,
    m.clean_title,
    m.year
HAVING COUNT(r.rating) >= 50
ORDER BY avg_rating DESC, total_ratings DESC
LIMIT 10;
"""

join_analysis = pd.read_sql_query(query, conn)

display(join_analysis)

,movieId,clean_title,year,total_ratings,avg_rating
0,318,"Shawshank Redemption, The",1994.0,317,4.43
1,858,"Godfather, The",1972.0,192,4.29
2,2959,Fight Club,1999.0,218,4.27
3,750,Dr. Strangelove or: How I Learned to Stop Worr...,1964.0,97,4.27
4,1276,Cool Hand Luke,1967.0,57,4.27
5,1221,"Godfather: Part II, The",1974.0,129,4.26
6,904,Rear Window,1954.0,84,4.26
7,1213,Goodfellas,1990.0,126,4.25
8,48516,"Departed, The",2006.0,107,4.25
9,50,"Usual Suspects, The",1995.0,204,4.24


## 5. Genre Analysis

Movie genres are normalized into a separate table to enable accurate genre-level analysis using SQL.

A Common Table Expression (CTE) is then used to calculate the number of movies, average rating, and total ratings for each genre.

In [ ]:
# Create normalized movie-genre table

movie_genres_df = (
    movies_df[["movieId", "genres"]]
    .assign(genre=movies_df["genres"].str.split("|"))
    .explode("genre")
    [["movieId", "genre"]]
)

movie_genres_df.to_sql(
    "movie_genres",
    conn,
    if_exists="replace",
    index=False
)

print("movie_genres table created successfully!")

display(movie_genres_df.head())

movie_genres table created successfully!


,movieId,genre
0,1,Adventure
0,1,Animation
0,1,Children
0,1,Comedy
0,1,Fantasy


In [ ]:
query = """
WITH genre_stats AS (
    SELECT
        mg.genre,
        COUNT(DISTINCT mg.movieId) AS total_movies,
        COUNT(r.rating) AS total_ratings,
        AVG(r.rating) AS avg_rating
    FROM movie_genres mg
    LEFT JOIN ratings r
        ON mg.movieId = r.movieId
    WHERE mg.genre != '(no genres listed)'
    GROUP BY mg.genre
)

SELECT
    genre,
    total_movies,
    total_ratings,
    ROUND(avg_rating, 2) AS average_rating
FROM genre_stats
ORDER BY total_ratings DESC;
"""

genre_analysis = pd.read_sql_query(query, conn)

display(genre_analysis)

,genre,total_movies,total_ratings,average_rating
0,Drama,4361,41928,3.66
1,Comedy,3756,39053,3.38
2,Action,1828,30635,3.45
3,Thriller,1894,26452,3.49
4,Adventure,1263,24161,3.51
5,Romance,1596,18124,3.51
6,Sci-Fi,980,17243,3.46
7,Crime,1199,16681,3.66
8,Fantasy,779,11834,3.49
9,Children,664,9208,3.41


## 6. Top-Rated Movie in Each Genre

A SQL window function is used to rank movies within each genre. Only movies with at least 20 ratings are considered to reduce the impact of movies with insufficient rating data.

In [ ]:
query = """
WITH ranked_movies AS (
    SELECT
        mg.genre,
        m.clean_title,
        m.year,
        m.average_rating,
        m.rating_count,

        ROW_NUMBER() OVER (
            PARTITION BY mg.genre
            ORDER BY m.average_rating DESC,
                     m.rating_count DESC
        ) AS movie_rank

    FROM movie_genres mg

    JOIN movies m
        ON mg.movieId = m.movieId

    WHERE
        m.rating_count >= 20
        AND mg.genre != '(no genres listed)'
)

SELECT
    genre,
    clean_title,
    year,
    average_rating,
    rating_count
FROM ranked_movies
WHERE movie_rank = 1
ORDER BY genre;
"""

top_by_genre = pd.read_sql_query(query, conn)

display(top_by_genre)

,genre,clean_title,year,average_rating,rating_count
0,Action,Logan,2017.0,4.28,25
1,Adventure,Lawrence of Arabia,1962.0,4.30,45
2,Animation,Spirited Away (Sen to Chihiro no kamikakushi),2001.0,4.16,87
3,Children,Toy Story 3,2010.0,4.11,55
4,Comedy,"Philadelphia Story, The",1940.0,4.31,29
5,Crime,"Shawshank Redemption, The",1994.0,4.43,317
6,Documentary,Hoop Dreams,1994.0,4.29,29
7,Drama,"Streetcar Named Desire, A",1951.0,4.47,20
8,Fantasy,"Princess Bride, The",1987.0,4.23,142
9,Film-Noir,Sunset Blvd. (a.k.a. Sunset Boulevard),1950.0,4.33,27


## 7. User Rating Activity Over Time

This analysis examines the number of ratings submitted over time to understand user activity patterns in the MovieLens dataset.

In [ ]:
query = """
SELECT
    CAST(strftime('%Y', datetime) AS INTEGER) AS rating_year,
    COUNT(*) AS total_ratings,
    COUNT(DISTINCT userId) AS active_users,
    ROUND(AVG(rating), 2) AS average_rating
FROM ratings
GROUP BY rating_year
ORDER BY rating_year;
"""

rating_trends = pd.read_sql_query(query, conn)

display(rating_trends)

,rating_year,total_ratings,active_users,average_rating
0,1996,6040,97,3.54
1,1997,1916,33,3.73
2,1998,507,11,3.44
3,1999,2439,27,3.77
4,2000,10061,56,3.39
5,2001,3922,38,3.51
6,2002,3478,25,3.61
7,2003,4014,31,3.50
8,2004,3279,24,3.51
9,2005,5813,43,3.43


## 8. Hidden Gem Identification

Hidden gems are defined as movies that have strong user ratings but relatively lower popularity.

For this initial SQL analysis, movies must:

- Have an average rating of at least 4.0
- Have between 20 and 100 user ratings

A more advanced Hidden Gem Score will be developed later in the recommendation system.

In [ ]:
query = """
SELECT
    movieId,
    clean_title,
    year,
    genres,
    rating_count,
    average_rating
FROM movies
WHERE
    average_rating >= 4.0
    AND rating_count BETWEEN 20 AND 100
ORDER BY
    average_rating DESC,
    rating_count DESC
LIMIT 20;
"""

hidden_gems = pd.read_sql_query(query, conn)

display(hidden_gems)

,movieId,clean_title,year,genres,rating_count,average_rating
0,1104,"Streetcar Named Desire, A",1951.0,Drama,20,4.47
1,922,Sunset Blvd. (a.k.a. Sunset Boulevard),1950.0,Drama|Film-Noir|Romance,27,4.33
2,898,"Philadelphia Story, The",1940.0,Comedy|Drama|Romance,29,4.31
3,1204,Lawrence of Arabia,1962.0,Adventure|Drama|War,45,4.30
4,475,In the Name of the Father,1993.0,Drama,25,4.30
5,246,Hoop Dreams,1994.0,Documentary,29,4.29
6,1235,Harold and Maude,1971.0,Comedy|Drama|Romance,26,4.29
7,168252,Logan,2017.0,Action|Sci-Fi,25,4.28
8,750,Dr. Strangelove or: How I Learned to Stop Worr...,1964.0,Comedy|War,97,4.27
9,1276,Cool Hand Luke,1967.0,Drama,57,4.27


## 9. User Rating Behavior

Users are analyzed based on their average rating behavior to understand whether they tend to rate movies strictly, moderately, or generously.

In [ ]:
query = """
WITH user_behavior AS (
    SELECT
        userId,
        COUNT(*) AS total_ratings,
        AVG(rating) AS avg_rating
    FROM ratings
    GROUP BY userId
)

SELECT
    userId,
    total_ratings,
    ROUND(avg_rating, 2) AS average_rating,

    CASE
        WHEN avg_rating >= 4.0 THEN 'Generous Rater'
        WHEN avg_rating >= 3.0 THEN 'Moderate Rater'
        ELSE 'Strict Rater'
    END AS rating_behavior

FROM user_behavior

ORDER BY total_ratings DESC
LIMIT 20;
"""

user_behavior = pd.read_sql_query(query, conn)

display(user_behavior)

,userId,total_ratings,average_rating,rating_behavior
0,414,2698,3.39,Moderate Rater
1,599,2478,2.64,Strict Rater
2,474,2108,3.40,Moderate Rater
3,448,1864,2.85,Strict Rater
4,274,1346,3.24,Moderate Rater
5,610,1302,3.69,Moderate Rater
6,68,1260,3.23,Moderate Rater
7,380,1218,3.67,Moderate Rater
8,606,1115,3.66,Moderate Rater
9,288,1055,3.15,Moderate Rater


## 10. Genre Popularity Ranking

Genres are ranked according to the total number of user ratings they received, providing insight into the most actively consumed movie categories.

In [ ]:
query = """
WITH genre_popularity AS (
    SELECT
        mg.genre,
        COUNT(r.rating) AS total_ratings
    FROM movie_genres mg

    JOIN ratings r
        ON mg.movieId = r.movieId

    WHERE mg.genre != '(no genres listed)'

    GROUP BY mg.genre
)

SELECT
    genre,
    total_ratings,

    RANK() OVER (
        ORDER BY total_ratings DESC
    ) AS popularity_rank

FROM genre_popularity

ORDER BY popularity_rank;
"""

genre_ranking = pd.read_sql_query(query, conn)

display(genre_ranking)

,genre,total_ratings,popularity_rank
0,Drama,41928,1
1,Comedy,39053,2
2,Action,30635,3
3,Thriller,26452,4
4,Adventure,24161,5
5,Romance,18124,6
6,Sci-Fi,17243,7
7,Crime,16681,8
8,Fantasy,11834,9
9,Children,9208,10


In [ ]:
movie_genres_df.to_csv(
    "movie_genres.csv",
    index=False
)

print("movie_genres.csv saved successfully!")

movie_genres.csv saved successfully!


## Key SQL Insights

- Identified the most popular and highly rated movies while applying minimum rating thresholds for reliable rankings.
- Analyzed user engagement and identified the most active users based on rating activity.
- Normalized movie genres into a separate relational table for accurate genre-level analysis.
- Analyzed genre popularity and average rating performance.
- Used SQL window functions to identify top-performing movies within each genre.
- Analyzed user rating activity over time and categorized users based on their rating behavior.
- Identified potential hidden gems based on high ratings and moderate popularity.
- Applied advanced SQL concepts including JOINs, CTEs, CASE statements, GROUP BY, HAVING, ROW_NUMBER(), RANK(), and PARTITION BY.